# CleanVision — Model Training Notebook

Train the CNN cleanliness classifier on Google Colab (free GPU).

## Steps
1. Upload your dataset images to Google Drive
2. Mount Drive and point to the dataset
3. Train the model
4. Download the trained `model.pth` and place it in `backend/model.pth`

In [ ]:
#@title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Configuration — set your dataset path
DATASET_PATH = '/content/drive/MyDrive/cleanvision_dataset'  #@param {type:"string"}
# Expected structure:
#   cleanvision_dataset/
#     clean/       (hundreds of clean room images)
#     dirty/       (hundreds of dirty room images)

import os
print('Dataset contents:')
for d in ['clean', 'dirty']:
    p = os.path.join(DATASET_PATH, d)
    if os.path.exists(p):
        count = len(os.listdir(p))
        print(f'  {d}/: {count} images')
    else:
        print(f'  {d}/: NOT FOUND')

In [ ]:
#@title 3. Install dependencies
!pip install torch torchvision pillow matplotlib

In [ ]:
#@title 4. Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
#@title 5. Define the CNN model (same architecture as backend)
class CleanlinessCNN(nn.Module):
    def __init__(self):
        super(CleanlinessCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = CleanlinessCNN().to(device)
print(model)

In [ ]:
#@title 6. Dataset class
class CleanlinessDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])
        self.samples = []
        for label_name, label_val in [('clean', 1.0), ('dirty', 0.0)]:
            folder = os.path.join(root_dir, label_name)
            if not os.path.exists(folder):
                continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(folder, fname), label_val))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

dataset = CleanlinessDataset(DATASET_PATH)
print(f'Total samples: {len(dataset)}')

In [ ]:
#@title 7. Split dataset and create dataloaders
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

In [ ]:
#@title 8. Training setup
EPOCHS = 25  #@param {type:"integer"}
LR = 0.001  #@param {type:"number"}

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [ ]:
#@title 9. Train the model
train_losses = []
val_losses = []
val_accs = []
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # Training
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = (outputs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    val_acc = correct / total if total > 0 else 0
    val_accs.append(val_acc)

    scheduler.step()

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/content/best_model.pth')

    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

print('\nTraining complete!')

In [ ]:
#@title 10. Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss', marker='o')
ax1.plot(val_losses, label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves')
ax1.legend()
ax1.grid(True)

ax2.plot(val_accs, label='Val Accuracy', marker='s', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
#@title 11. Download the trained model
from google.colab import files
print('Downloading best_model.pth...')
files.download('/content/best_model.pth')
print('\nSave this file as: backend/model.pth')